In [3]:
customers = [

(1, "Rahul", "Hyderabad"),
(2, "Priya", "Bangalore"),
(3, "Amit", "Mumbai"),
(4, "Sneha", "Chennai"),
(5, "Farhan", "Delhi")

]

customer_columns = [
    "customer_id",
    "customer_name",
    "city"
]

customers_df = spark.createDataFrame(
    customers,
    customer_columns
)

customers_df.show()




+-----------+-------------+---------+
|customer_id|customer_name|     city|
+-----------+-------------+---------+
|          1|        Rahul|Hyderabad|
|          2|        Priya|Bangalore|
|          3|         Amit|   Mumbai|
|          4|        Sneha|  Chennai|
|          5|       Farhan|    Delhi|
+-----------+-------------+---------+



In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CustomerApp") \
    .getOrCreate()

In [4]:
orders = [

(101, 1, "Laptop", 65000),
(102, 2, "Mobile", 25000),
(103, 1, "TV", 45000),
(104, 3, "Chair", 5000),
(105, 7, "Watch", 8000)

]

order_columns = [
    "order_id",
    "customer_id",
    "product",
    "amount"
]

orders_df = spark.createDataFrame(
    orders,
    order_columns
)

In [5]:
# Inner Join
customers_df.join(
    orders_df,
    "customer_id",
    "inner"
).show()


+-----------+-------------+---------+--------+-------+------+
|customer_id|customer_name|     city|order_id|product|amount|
+-----------+-------------+---------+--------+-------+------+
|          1|        Rahul|Hyderabad|     101| Laptop| 65000|
|          1|        Rahul|Hyderabad|     103|     TV| 45000|
|          2|        Priya|Bangalore|     102| Mobile| 25000|
|          3|         Amit|   Mumbai|     104|  Chair|  5000|
+-----------+-------------+---------+--------+-------+------+



In [6]:
# Left Join
customers_df.join(
    orders_df,
    "customer_id",
    "left"
).show()

+-----------+-------------+---------+--------+-------+------+
|customer_id|customer_name|     city|order_id|product|amount|
+-----------+-------------+---------+--------+-------+------+
|          1|        Rahul|Hyderabad|     103|     TV| 45000|
|          1|        Rahul|Hyderabad|     101| Laptop| 65000|
|          2|        Priya|Bangalore|     102| Mobile| 25000|
|          5|       Farhan|    Delhi|    NULL|   NULL|  NULL|
|          3|         Amit|   Mumbai|     104|  Chair|  5000|
|          4|        Sneha|  Chennai|    NULL|   NULL|  NULL|
+-----------+-------------+---------+--------+-------+------+



In [7]:

# Right Join
customers_df.join(
    orders_df,
    "customer_id",
    "right"
).show()

+-----------+-------------+---------+--------+-------+------+
|customer_id|customer_name|     city|order_id|product|amount|
+-----------+-------------+---------+--------+-------+------+
|          1|        Rahul|Hyderabad|     101| Laptop| 65000|
|          2|        Priya|Bangalore|     102| Mobile| 25000|
|          7|         NULL|     NULL|     105|  Watch|  8000|
|          1|        Rahul|Hyderabad|     103|     TV| 45000|
|          3|         Amit|   Mumbai|     104|  Chair|  5000|
+-----------+-------------+---------+--------+-------+------+



In [8]:
customers_df.join(
    orders_df,
    "customer_id",
    "full"
).show()

+-----------+-------------+---------+--------+-------+------+
|customer_id|customer_name|     city|order_id|product|amount|
+-----------+-------------+---------+--------+-------+------+
|          1|        Rahul|Hyderabad|     101| Laptop| 65000|
|          1|        Rahul|Hyderabad|     103|     TV| 45000|
|          2|        Priya|Bangalore|     102| Mobile| 25000|
|          3|         Amit|   Mumbai|     104|  Chair|  5000|
|          4|        Sneha|  Chennai|    NULL|   NULL|  NULL|
|          5|       Farhan|    Delhi|    NULL|   NULL|  NULL|
|          7|         NULL|     NULL|     105|  Watch|  8000|
+-----------+-------------+---------+--------+-------+------+



In [9]:
from pyspark.sql.functions import sum

customers_df.join(
    orders_df,
    "customer_id"
).groupBy(
    "customer_name"
).agg(
    sum("amount").alias("total_spent")
).show()

+-------------+-----------+
|customer_name|total_spent|
+-------------+-----------+
|        Priya|      25000|
|        Rahul|     110000|
|         Amit|       5000|
+-------------+-----------+



In [10]:
employees = [

(101,"Rahul","IT",75000),
(102,"Priya","IT",85000),
(103,"Amit","IT",65000),

(104,"Sneha","HR",70000),
(105,"Farhan","HR",90000),

(106,"Neha","Finance",95000),
(107,"Arjun","Finance",80000),
(108,"Meera","Finance",75000)

]

columns = [
    "employee_id",
    "employee_name",
    "department",
    "salary"
]

df = spark.createDataFrame(
    employees,
    columns
)

df.show()


+-----------+-------------+----------+------+
|employee_id|employee_name|department|salary|
+-----------+-------------+----------+------+
|        101|        Rahul|        IT| 75000|
|        102|        Priya|        IT| 85000|
|        103|         Amit|        IT| 65000|
|        104|        Sneha|        HR| 70000|
|        105|       Farhan|        HR| 90000|
|        106|         Neha|   Finance| 95000|
|        107|        Arjun|   Finance| 80000|
|        108|        Meera|   Finance| 75000|
+-----------+-------------+----------+------+



In [13]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col



In [15]:
window_spec = Window.orderBy(col("salary").desc())

df.withColumn(
    "row_number",
    row_number().over(window_spec)
).show()

+-----------+-------------+----------+------+----------+
|employee_id|employee_name|department|salary|row_number|
+-----------+-------------+----------+------+----------+
|        106|         Neha|   Finance| 95000|         1|
|        105|       Farhan|        HR| 90000|         2|
|        102|        Priya|        IT| 85000|         3|
|        107|        Arjun|   Finance| 80000|         4|
|        101|        Rahul|        IT| 75000|         5|
|        108|        Meera|   Finance| 75000|         6|
|        104|        Sneha|        HR| 70000|         7|
|        103|         Amit|        IT| 65000|         8|
+-----------+-------------+----------+------+----------+



In [20]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, col

window_spec = Window.orderBy(
    col("salary").desc()
)

df.withColumn(
    "rank",
    rank().over(window_spec)
).show()

+-----------+-------------+----------+------+----+
|employee_id|employee_name|department|salary|rank|
+-----------+-------------+----------+------+----+
|        106|         Neha|   Finance| 95000|   1|
|        105|       Farhan|        HR| 90000|   2|
|        102|        Priya|        IT| 85000|   3|
|        107|        Arjun|   Finance| 80000|   4|
|        101|        Rahul|        IT| 75000|   5|
|        108|        Meera|   Finance| 75000|   5|
|        104|        Sneha|        HR| 70000|   7|
|        103|         Amit|        IT| 65000|   8|
+-----------+-------------+----------+------+----+



In [21]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank, col

window_spec = Window.orderBy(
    col("salary").desc()
)

df.withColumn(
    "dense_rank",
    dense_rank().over(window_spec)
).show()

+-----------+-------------+----------+------+----------+
|employee_id|employee_name|department|salary|dense_rank|
+-----------+-------------+----------+------+----------+
|        106|         Neha|   Finance| 95000|         1|
|        105|       Farhan|        HR| 90000|         2|
|        102|        Priya|        IT| 85000|         3|
|        107|        Arjun|   Finance| 80000|         4|
|        101|        Rahul|        IT| 75000|         5|
|        108|        Meera|   Finance| 75000|         5|
|        104|        Sneha|        HR| 70000|         6|
|        103|         Amit|        IT| 65000|         7|
+-----------+-------------+----------+------+----------+



In [22]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, col

window_spec = Window.partitionBy(
    "department"
).orderBy(
    col("salary").desc()
)

df.withColumn(
    "department_rank",
    rank().over(window_spec)
).show()

+-----------+-------------+----------+------+---------------+
|employee_id|employee_name|department|salary|department_rank|
+-----------+-------------+----------+------+---------------+
|        106|         Neha|   Finance| 95000|              1|
|        107|        Arjun|   Finance| 80000|              2|
|        108|        Meera|   Finance| 75000|              3|
|        105|       Farhan|        HR| 90000|              1|
|        104|        Sneha|        HR| 70000|              2|
|        102|        Priya|        IT| 85000|              1|
|        101|        Rahul|        IT| 75000|              2|
|        103|         Amit|        IT| 65000|              3|
+-----------+-------------+----------+------+---------------+



In [23]:
%%writefile patients.csv
patient_id,patient_name,city,age,gender,blood_group,insurance_status
101,Rahul Sharma,Hyderabad,35,Male,O+,Active
102,Priya Reddy,Bangalore,29,Female,A+,Active
103,Amit Kumar,Mumbai,42,Male,B+,Inactive
104,Sneha Patel,Chennai,31,Female,O+,Active
105,Farhan Ali,Delhi,55,Male,AB+,Active
106,Neha Singh,,38,Female,A+,Inactive
107,Arjun Verma,Pune,26,Male,B+,Active
108,Meera Nair,Kochi,48,Female,O-,Active
109,Kiran Rao,Hyderabad,33,Male,,Inactive
110,Nisha Reddy,Bangalore,41,Female,A+,Active

Writing patients.csv


In [24]:
%%writefile appointments.csv
appointment_id,patient_id,doctor_name,department,appointment_date,consult
5001,101,Dr. Ramesh,Cardiology,2025-01-10,1500,Completed
5002,102,Dr. Suresh,Neurology,2025-01-11,2000,Completed
5003,101,Dr. Anita,Dermatology,2025-01-15,1000,Completed
5004,103,Dr. Ramesh,Cardiology,2025-01-20,1500,Cancelled
5005,104,Dr. Priya,Orthopedics,2500,2500,Completed
5006,105,Dr. Anita,Dermatology,2025-01-25,1000,Pending
5007,107,Dr. Suresh,Neurology,2025-02-01,2000,Completed
5008,110,Dr. Priya,Orthopedics,2025-02-03,2500,Completed
5009,120,Dr. Ramesh,Cardiology,2025-02-05,1500,Completed
5010,108,Dr. Anita,Dermatology,2025-02-10,,Pending

Writing appointments.csv


In [25]:
%%writefile patient_preferences.json
[
{
"patient_id":101,
"preferred_hospital":"Apollo",
"contact":{
"phone":"9876500011",
"email":"rahul@gmail.com"
}
},
{
"patient_id":102,
"preferred_hospital":"Yashoda",
"contact":{
"phone":null,
"email":"priya@gmail.com"
}
},
{
"patient_id":103,
"preferred_hospital":"Care",
"contact":{
"phone":"9876500013",
"email":null
}
},
{
"patient_id":104,
"preferred_hospital":null,
"contact":{
"phone":"9876500014",
"email":"sneha@gmail.com"
}

}
]

Writing patient_preferences.json


In [26]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("HospitalDataAnalysis") \
    .getOrCreate()

In [27]:
patients_df = spark.read.csv(
    "patients.csv",
    header=True,
    inferSchema=True
)

patients_df.show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+

In [28]:
appointments_df = spark.read.csv(
    "appointments.csv",
    header=True,
    inferSchema=True
)

appointments_df.show()

+--------------+----------+-----------+-----------+-------------------+-------+
|appointment_id|patient_id|doctor_name| department|   appointment_date|consult|
+--------------+----------+-----------+-----------+-------------------+-------+
|          5001|       101| Dr. Ramesh| Cardiology|2025-01-10 00:00:00|   1500|
|          5002|       102| Dr. Suresh|  Neurology|2025-01-11 00:00:00|   2000|
|          5003|       101|  Dr. Anita|Dermatology|2025-01-15 00:00:00|   1000|
|          5004|       103| Dr. Ramesh| Cardiology|2025-01-20 00:00:00|   1500|
|          5005|       104|  Dr. Priya|Orthopedics|2500-01-01 00:00:00|   2500|
|          5006|       105|  Dr. Anita|Dermatology|2025-01-25 00:00:00|   1000|
|          5007|       107| Dr. Suresh|  Neurology|2025-02-01 00:00:00|   2000|
|          5008|       110|  Dr. Priya|Orthopedics|2025-02-03 00:00:00|   2500|
|          5009|       120| Dr. Ramesh| Cardiology|2025-02-05 00:00:00|   1500|
|          5010|       108|  Dr. Anita|D

In [31]:
patients_df.printSchema()

root
 |-- patient_id: integer (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- blood_group: string (nullable = true)
 |-- insurance_status: string (nullable = true)



In [32]:
patients_df.count()

10

In [33]:
patients_df.show(5)

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
+----------+------------+---------+---+------+-----------+----------------+
only showing top 5 rows


In [34]:
patients_df.select("city").distinct().show()

+---------+
|     city|
+---------+
|Bangalore|
|    Kochi|
|  Chennai|
|   Mumbai|
|     Pune|
|    Delhi|
|Hyderabad|
|     NULL|
+---------+



In [35]:
appointments_df.select("department").distinct().show()

+-----------+
| department|
+-----------+
|  Neurology|
|Dermatology|
| Cardiology|
|Orthopedics|
+-----------+



In [36]:
patients_df.write.mode("overwrite").parquet("patients_parquet")

In [37]:
patients_parquet_df = spark.read.parquet("patients_parquet")

In [38]:
csv_count = patients_df.count()
parquet_count = patients_parquet_df.count()

print("CSV Record Count:", csv_count)
print("Parquet Record Count:", parquet_count)

CSV Record Count: 10
Parquet Record Count: 10


In [39]:
patients_df.filter(
    patients_df.city == "Hyderabad"
).show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
+----------+------------+---------+---+------+-----------+----------------+



In [40]:
patients_df.filter(
    patients_df.gender == "Female"
).show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [41]:
patients_df.filter(
    patients_df.age > 40
).show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [42]:
appointments_df.filter(
    appointments_df.status == "Completed"
).show()

PySparkAttributeError: [ATTRIBUTE_NOT_SUPPORTED] Attribute `status` is not supported.

In [43]:
appointments_df.columns

['appointment_id',
 'patient_id',
 'doctor_name',
 'department',
 'appointment_date',
 'consult']

In [44]:
%%writefile appointments.csv
appointment_id,patient_id,doctor_name,department,appointment_date,consultation_fee,status
5001,101,Dr. Ramesh,Cardiology,2025-01-10,1500,Completed
5002,102,Dr. Suresh,Neurology,2025-01-11,2000,Completed
5003,101,Dr. Anita,Dermatology,2025-01-15,1000,Completed
5004,103,Dr. Ramesh,Cardiology,2025-01-20,1500,Cancelled
5005,104,Dr. Priya,Orthopedics,2025-01-22,2500,Completed
5006,105,Dr. Anita,Dermatology,2025-01-25,1000,Pending
5007,107,Dr. Suresh,Neurology,2025-02-01,2000,Completed
5008,110,Dr. Priya,Orthopedics,2025-02-03,2500,Completed
5009,120,Dr. Ramesh,Cardiology,2025-02-05,1500,Completed
5010,108,Dr. Anita,Dermatology,2025-02-10,,Pending

Overwriting appointments.csv


In [45]:
appointments_df = spark.read.csv(
    "appointments.csv",
    header=True,
    inferSchema=True
)

In [46]:
appointments_df.columns

['appointment_id',
 'patient_id',
 'doctor_name',
 'department',
 'appointment_date',
 'consultation_fee',
 'status']

In [47]:
appointments_df.filter(
    appointments_df.status == "Completed"
).show()

+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|          5001|       101| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|          5003|       101|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|
|          5005|       104|  Dr. Priya|Orthopedics|      2025-01-22|            2500|Completed|
|          5007|       107| Dr. Suresh|  Neurology|      2025-02-01|            2000|Completed|
|          5008|       110|  Dr. Priya|Orthopedics|      2025-02-03|            2500|Completed|
|          5009|       120| Dr. Ramesh| Cardiology|      2025-02-05|            1500|Completed|
+--------------+----------+-----------+-

In [48]:
appointments_df.filter(
    appointments_df.status == "Pending"
).show()

+--------------+----------+-----------+-----------+----------------+----------------+-------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee| status|
+--------------+----------+-----------+-----------+----------------+----------------+-------+
|          5006|       105|  Dr. Anita|Dermatology|      2025-01-25|            1000|Pending|
|          5010|       108|  Dr. Anita|Dermatology|      2025-02-10|            NULL|Pending|
+--------------+----------+-----------+-----------+----------------+----------------+-------+



In [49]:
appointments_df.filter(
    appointments_df.consultation_fee > 1500
).show()

+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|          5005|       104|  Dr. Priya|Orthopedics|      2025-01-22|            2500|Completed|
|          5007|       107| Dr. Suresh|  Neurology|      2025-02-01|            2000|Completed|
|          5008|       110|  Dr. Priya|Orthopedics|      2025-02-03|            2500|Completed|
+--------------+----------+-----------+-----------+----------------+----------------+---------+



In [50]:
patients_df.filter(
    patients_df.insurance_status == "Active"
).show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [51]:
patients_df.filter(
    patients_df.insurance_status == "Active"
).show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [52]:
patients_df.filter(
    patients_df.insurance_status == "Inactive"
).show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
+----------+------------+---------+---+------+-----------+----------------+



In [53]:
patients_df.filter(
    patients_df.blood_group == "O+"
).show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [54]:
appointments_df.filter(
    appointments_df.department == "Cardiology"
).show()

+--------------+----------+-----------+----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name|department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+----------+----------------+----------------+---------+
|          5001|       101| Dr. Ramesh|Cardiology|      2025-01-10|            1500|Completed|
|          5004|       103| Dr. Ramesh|Cardiology|      2025-01-20|            1500|Cancelled|
|          5009|       120| Dr. Ramesh|Cardiology|      2025-02-05|            1500|Completed|
+--------------+----------+-----------+----------+----------------+----------------+---------+



In [55]:
from pyspark.sql.functions import col

patients_df.filter(
    col("city").isNull()
).show()

+----------+------------+----+---+------+-----------+----------------+
|patient_id|patient_name|city|age|gender|blood_group|insurance_status|
+----------+------------+----+---+------+-----------+----------------+
|       106|  Neha Singh|NULL| 38|Female|         A+|        Inactive|
+----------+------------+----+---+------+-----------+----------------+



In [56]:
patients_df.filter(
    col("blood_group").isNull()
).show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
+----------+------------+---------+---+------+-----------+----------------+



In [57]:
appointments_df.filter(
    col("consultation_fee").isNull()
).show()

+--------------+----------+-----------+-----------+----------------+----------------+-------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee| status|
+--------------+----------+-----------+-----------+----------------+----------------+-------+
|          5010|       108|  Dr. Anita|Dermatology|      2025-02-10|            NULL|Pending|
+--------------+----------+-----------+-----------+----------------+----------------+-------+



In [58]:
from pyspark.sql.functions import count, when

patients_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in patients_df.columns
]).show()

+----------+------------+----+---+------+-----------+----------------+
|patient_id|patient_name|city|age|gender|blood_group|insurance_status|
+----------+------------+----+---+------+-----------+----------------+
|         0|           0|   1|  0|     0|          1|               0|
+----------+------------+----+---+------+-----------+----------------+



In [59]:
patients_df = patients_df.fillna({
    "city": "Unknown"
})

patients_df.show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       106|  Neha Singh|  Unknown| 38|Female|         A+|        Inactive|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+

In [60]:
patients_df = patients_df.fillna({
    "blood_group": "Not Available"
})

patients_df.show()

+----------+------------+---------+---+------+-------------+----------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|
+----------+------------+---------+---+------+-------------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|           O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|          AB+|          Active|
|       106|  Neha Singh|  Unknown| 38|Female|           A+|        Inactive|
|       107| Arjun Verma|     Pune| 26|  Male|           B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|           O-|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|Not Available|        Inactive|
|       110| Nisha Reddy|Bangalore| 41|Female|           A+|    

In [61]:
appointments_df = appointments_df.fillna({
    "consultation_fee": 0
})

appointments_df.show()

+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|          5001|       101| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|          5003|       101|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|
|          5004|       103| Dr. Ramesh| Cardiology|      2025-01-20|            1500|Cancelled|
|          5005|       104|  Dr. Priya|Orthopedics|      2025-01-22|            2500|Completed|
|          5006|       105|  Dr. Anita|Dermatology|      2025-01-25|            1000|  Pending|
|          5007|       107| Dr. Suresh|  Neurology|      2025-02-01|            2000|Completed|
|          5008|       110|  Dr. Priya|O

In [62]:
appointments_df.dropna(
    subset=["consultation_fee"]
).show()

+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|          5001|       101| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|          5003|       101|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|
|          5004|       103| Dr. Ramesh| Cardiology|      2025-01-20|            1500|Cancelled|
|          5005|       104|  Dr. Priya|Orthopedics|      2025-01-22|            2500|Completed|
|          5006|       105|  Dr. Anita|Dermatology|      2025-01-25|            1000|  Pending|
|          5007|       107| Dr. Suresh|  Neurology|      2025-02-01|            2000|Completed|
|          5008|       110|  Dr. Priya|O

In [63]:
from pyspark.sql.functions import when

patients_df = patients_df.withColumn(
    "data_quality_status",
    when(
        col("city").isNull() | col("blood_group").isNull(),
        "Incomplete"
    ).otherwise("Complete")
)

patients_df.show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|
+----------+------------+---------+---+------+-------------+----------------+-------------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|           Complete|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|           Complete|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|        Inactive|           Complete|
|       104| Sneha Patel|  Chennai| 31|Female|           O+|          Active|           Complete|
|       105|  Farhan Ali|    Delhi| 55|  Male|          AB+|          Active|           Complete|
|       106|  Neha Singh|  Unknown| 38|Female|           A+|        Inactive|           Complete|
|       107| Arjun Verma|     Pune| 26|  Male|           B+|          Active|           Complete|
|       108|  Meera 

In [64]:
patients_df.groupBy(
    "data_quality_status"
).count().show()

+-------------------+-----+
|data_quality_status|count|
+-------------------+-----+
|           Complete|   10|
+-------------------+-----+



In [65]:
from pyspark.sql.functions import upper, lower, length, substring, when, concat_ws, trim, col

In [66]:
patients_df.select(
    upper(col("patient_name")).alias("patient_name_upper")
).show()

+------------------+
|patient_name_upper|
+------------------+
|      RAHUL SHARMA|
|       PRIYA REDDY|
|        AMIT KUMAR|
|       SNEHA PATEL|
|        FARHAN ALI|
|        NEHA SINGH|
|       ARJUN VERMA|
|        MEERA NAIR|
|         KIRAN RAO|
|       NISHA REDDY|
+------------------+



In [67]:
patients_df.select(
    lower(col("patient_name")).alias("patient_name_lower")
).show()

+------------------+
|patient_name_lower|
+------------------+
|      rahul sharma|
|       priya reddy|
|        amit kumar|
|       sneha patel|
|        farhan ali|
|        neha singh|
|       arjun verma|
|        meera nair|
|         kiran rao|
|       nisha reddy|
+------------------+



In [68]:
patients_df.select(
    "patient_name",
    length(col("patient_name")).alias("name_length")
).show()

+------------+-----------+
|patient_name|name_length|
+------------+-----------+
|Rahul Sharma|         12|
| Priya Reddy|         11|
|  Amit Kumar|         10|
| Sneha Patel|         11|
|  Farhan Ali|         10|
|  Neha Singh|         10|
| Arjun Verma|         11|
|  Meera Nair|         10|
|   Kiran Rao|          9|
| Nisha Reddy|         11|
+------------+-----------+



In [69]:
patients_df.select(
    "patient_name",
    substring(col("patient_name"), 1, 3).alias("first_3_letters")
).show()

+------------+---------------+
|patient_name|first_3_letters|
+------------+---------------+
|Rahul Sharma|            Rah|
| Priya Reddy|            Pri|
|  Amit Kumar|            Ami|
| Sneha Patel|            Sne|
|  Farhan Ali|            Far|
|  Neha Singh|            Neh|
| Arjun Verma|            Arj|
|  Meera Nair|            Mee|
|   Kiran Rao|            Kir|
| Nisha Reddy|            Nis|
+------------+---------------+



In [70]:
patients_df = patients_df.withColumn(
    "age_group",
    when(col("age") < 30, "Young")
    .when(col("age") < 50, "Adult")
    .otherwise("Senior")
)

patients_df.show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|age_group|
+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|           Complete|    Adult|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|           Complete|    Young|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|        Inactive|           Complete|    Adult|
|       104| Sneha Patel|  Chennai| 31|Female|           O+|          Active|           Complete|    Adult|
|       105|  Farhan Ali|    Delhi| 55|  Male|          AB+|          Active|           Complete|   Senior|
|       106|  Neha Singh|  Unknown| 38|Female|           A+|        Inactive|           Complete|    Adult|
|       107| Arjun Verma|   

In [71]:
patients_df = patients_df.withColumn(
    "insurance_flag",
    when(col("insurance_status") == "Active", 1)
    .otherwise(0)
)

patients_df.show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|age_group|insurance_flag|
+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|           Complete|    Adult|             1|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|           Complete|    Young|             1|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|        Inactive|           Complete|    Adult|             0|
|       104| Sneha Patel|  Chennai| 31|Female|           O+|          Active|           Complete|    Adult|             1|
|       105|  Farhan Ali|    Delhi| 55|  Male|          AB+|          Active|           Complete|   Senior|             1|
|       106|  Ne

In [72]:
patients_df = patients_df.withColumn(
    "senior_citizen",
    when(col("age") >= 60, "Yes")
    .otherwise("No")
)

patients_df.show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|
+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|           Complete|    Adult|             1|            No|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|           Complete|    Young|             1|            No|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|        Inactive|           Complete|    Adult|             0|            No|
|       104| Sneha Patel|  Chennai| 31|Female|           O+|          Active|           Complete|    Adult|             1|            No|
|       105|  Farhan Ali|    Delhi

In [73]:
patients_df.select(
    concat_ws(" - ", col("patient_name"), col("city")).alias("patient_city")
).show()

+--------------------+
|        patient_city|
+--------------------+
|Rahul Sharma - Hy...|
|Priya Reddy - Ban...|
| Amit Kumar - Mumbai|
|Sneha Patel - Che...|
|  Farhan Ali - Delhi|
|Neha Singh - Unknown|
|  Arjun Verma - Pune|
|  Meera Nair - Kochi|
|Kiran Rao - Hyder...|
|Nisha Reddy - Ban...|
+--------------------+



In [74]:
patients_df = patients_df.withColumn(
    "patient_name",
    trim(col("patient_name"))
)

patients_df.show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|
+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|           Complete|    Adult|             1|            No|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|           Complete|    Young|             1|            No|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|        Inactive|           Complete|    Adult|             0|            No|
|       104| Sneha Patel|  Chennai| 31|Female|           O+|          Active|           Complete|    Adult|             1|            No|
|       105|  Farhan Ali|    Delhi

In [75]:
patients_df = patients_df.withColumn(
    "patient_name",
    trim(col("patient_name"))
)

patients_df.show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|
+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|           Complete|    Adult|             1|            No|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|           Complete|    Young|             1|            No|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|        Inactive|           Complete|    Adult|             0|            No|
|       104| Sneha Patel|  Chennai| 31|Female|           O+|          Active|           Complete|    Adult|             1|            No|
|       105|  Farhan Ali|    Delhi

In [76]:
patients_df = patients_df.withColumn(
    "city",
    upper(col("city"))
)

patients_df.show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|
+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+
|       101|Rahul Sharma|HYDERABAD| 35|  Male|           O+|          Active|           Complete|    Adult|             1|            No|
|       102| Priya Reddy|BANGALORE| 29|Female|           A+|          Active|           Complete|    Young|             1|            No|
|       103|  Amit Kumar|   MUMBAI| 42|  Male|           B+|        Inactive|           Complete|    Adult|             0|            No|
|       104| Sneha Patel|  CHENNAI| 31|Female|           O+|          Active|           Complete|    Adult|             1|            No|
|       105|  Farhan Ali|    DELHI

In [77]:
from pyspark.sql.functions import count, avg, max, min, sum, desc

In [78]:
patients_df.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|    KOCHI|    1|
|BANGALORE|    2|
|    DELHI|    1|
|HYDERABAD|    2|
|  UNKNOWN|    1|
|  CHENNAI|    1|
|     PUNE|    1|
|   MUMBAI|    1|
+---------+-----+



In [79]:
patients_df.groupBy("gender").count().show()

+------+-----+
|gender|count|
+------+-----+
|Female|    5|
|  Male|    5|
+------+-----+



In [80]:
patients_df.groupBy("blood_group").count().show()

+-------------+-----+
|  blood_group|count|
+-------------+-----+
|          AB+|    1|
|           O+|    2|
|           O-|    1|
|           B+|    2|
|           A+|    3|
|Not Available|    1|
+-------------+-----+



In [81]:
appointments_df.groupBy("department").count().show()

+-----------+-----+
| department|count|
+-----------+-----+
|  Neurology|    2|
|Dermatology|    3|
| Cardiology|    3|
|Orthopedics|    2|
+-----------+-----+



In [82]:
patients_df.groupBy("city").agg(
    avg("age").alias("average_age")
).show()

+---------+-----------+
|     city|average_age|
+---------+-----------+
|    KOCHI|       48.0|
|BANGALORE|       35.0|
|    DELHI|       55.0|
|HYDERABAD|       34.0|
|  UNKNOWN|       38.0|
|  CHENNAI|       31.0|
|     PUNE|       26.0|
|   MUMBAI|       42.0|
+---------+-----------+



In [83]:
patients_df.groupBy("city").agg(
    avg("age").alias("average_age")
).show()

+---------+-----------+
|     city|average_age|
+---------+-----------+
|    KOCHI|       48.0|
|BANGALORE|       35.0|
|    DELHI|       55.0|
|HYDERABAD|       34.0|
|  UNKNOWN|       38.0|
|  CHENNAI|       31.0|
|     PUNE|       26.0|
|   MUMBAI|       42.0|
+---------+-----------+



In [84]:
patients_df.groupBy("city").agg(
    max("age").alias("maximum_age")
).show()

+---------+-----------+
|     city|maximum_age|
+---------+-----------+
|    KOCHI|         48|
|BANGALORE|         41|
|    DELHI|         55|
|HYDERABAD|         35|
|  UNKNOWN|         38|
|  CHENNAI|         31|
|     PUNE|         26|
|   MUMBAI|         42|
+---------+-----------+



In [85]:
patients_df.groupBy("city").agg(
    min("age").alias("minimum_age")
).show()

+---------+-----------+
|     city|minimum_age|
+---------+-----------+
|    KOCHI|         48|
|BANGALORE|         29|
|    DELHI|         55|
|HYDERABAD|         33|
|  UNKNOWN|         38|
|  CHENNAI|         31|
|     PUNE|         26|
|   MUMBAI|         42|
+---------+-----------+



In [86]:
appointments_df.groupBy("department").agg(
    avg("consultation_fee").alias("average_fee")
).show()

+-----------+-----------------+
| department|      average_fee|
+-----------+-----------------+
|  Neurology|           2000.0|
|Dermatology|666.6666666666666|
| Cardiology|           1500.0|
|Orthopedics|           2500.0|
+-----------+-----------------+



In [87]:
appointments_df.groupBy("department").agg(
    avg("consultation_fee").alias("average_fee")
).show()

+-----------+-----------------+
| department|      average_fee|
+-----------+-----------------+
|  Neurology|           2000.0|
|Dermatology|666.6666666666666|
| Cardiology|           1500.0|
|Orthopedics|           2500.0|
+-----------+-----------------+



In [88]:
appointments_df.groupBy("department").agg(
    sum("consultation_fee").alias("total_fee")
).show()

+-----------+---------+
| department|total_fee|
+-----------+---------+
|  Neurology|     4000|
|Dermatology|     2000|
| Cardiology|     4500|
|Orthopedics|     5000|
+-----------+---------+



In [89]:
appointments_df.groupBy("department") \
    .agg(sum("consultation_fee").alias("total_revenue")) \
    .orderBy(desc("total_revenue")) \
    .show(1)

+-----------+-------------+
| department|total_revenue|
+-----------+-------------+
|Orthopedics|         5000|
+-----------+-------------+
only showing top 1 row


In [90]:
inner_join_df = patients_df.join(
    appointments_df,
    on="patient_id",
    how="inner"
)

inner_join_df.show()

+----------+------------+---------+---+------+-----------+----------------+-------------------+---------+--------------+--------------+--------------+-----------+-----------+----------------+----------------+---------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|appointment_id|doctor_name| department|appointment_date|consultation_fee|   status|
+----------+------------+---------+---+------+-----------+----------------+-------------------+---------+--------------+--------------+--------------+-----------+-----------+----------------+----------------+---------+
|       101|Rahul Sharma|HYDERABAD| 35|  Male|         O+|          Active|           Complete|    Adult|             1|            No|          5003|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|
|       101|Rahul Sharma|HYDERABAD| 35|  Male|         O+|          Active|           Complete|    Adult|             1|    

In [91]:
left_join_df = patients_df.join(
    appointments_df,
    on="patient_id",
    how="left"
)

left_join_df.show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+--------------+-----------+-----------+----------------+----------------+---------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|appointment_id|doctor_name| department|appointment_date|consultation_fee|   status|
+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+--------------+-----------+-----------+----------------+----------------+---------+
|       101|Rahul Sharma|HYDERABAD| 35|  Male|           O+|          Active|           Complete|    Adult|             1|            No|          5003|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|
|       101|Rahul Sharma|HYDERABAD| 35|  Male|           O+|          Active|           Complete|    Adult|         

In [92]:
right_join_df = patients_df.join(
    appointments_df,
    on="patient_id",
    how="right"
)

right_join_df.show()

+----------+------------+---------+----+------+-----------+----------------+-------------------+---------+--------------+--------------+--------------+-----------+-----------+----------------+----------------+---------+
|patient_id|patient_name|     city| age|gender|blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|appointment_id|doctor_name| department|appointment_date|consultation_fee|   status|
+----------+------------+---------+----+------+-----------+----------------+-------------------+---------+--------------+--------------+--------------+-----------+-----------+----------------+----------------+---------+
|       101|Rahul Sharma|HYDERABAD|  35|  Male|         O+|          Active|           Complete|    Adult|             1|            No|          5001| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|       102| Priya Reddy|BANGALORE|  29|Female|         A+|          Active|           Complete|    Young|             1

In [93]:
full_join_df = patients_df.join(
    appointments_df,
    on="patient_id",
    how="full"
)

full_join_df.show()

+----------+------------+---------+----+------+-------------+----------------+-------------------+---------+--------------+--------------+--------------+-----------+-----------+----------------+----------------+---------+
|patient_id|patient_name|     city| age|gender|  blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|appointment_id|doctor_name| department|appointment_date|consultation_fee|   status|
+----------+------------+---------+----+------+-------------+----------------+-------------------+---------+--------------+--------------+--------------+-----------+-----------+----------------+----------------+---------+
|       101|Rahul Sharma|HYDERABAD|  35|  Male|           O+|          Active|           Complete|    Adult|             1|            No|          5001| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|       101|Rahul Sharma|HYDERABAD|  35|  Male|           O+|          Active|           Complete|    Adult|    

In [94]:
patients_df.join(
    appointments_df,
    on="patient_id",
    how="left_anti"
).show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|
+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+
|       106|  Neha Singh|  UNKNOWN| 38|Female|           A+|        Inactive|           Complete|    Adult|             0|            No|
|       109|   Kiran Rao|HYDERABAD| 33|  Male|Not Available|        Inactive|           Complete|    Adult|             0|            No|
+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+



In [95]:
appointments_df.join(
    patients_df,
    on="patient_id",
    how="left_anti"
).show()

+----------+--------------+-----------+----------+----------------+----------------+---------+
|patient_id|appointment_id|doctor_name|department|appointment_date|consultation_fee|   status|
+----------+--------------+-----------+----------+----------------+----------------+---------+
|       120|          5009| Dr. Ramesh|Cardiology|      2025-02-05|            1500|Completed|
+----------+--------------+-----------+----------+----------------+----------------+---------+



In [96]:
appointments_df.groupBy("patient_id") \
    .count() \
    .show()

+----------+-----+
|patient_id|count|
+----------+-----+
|       108|    1|
|       101|    2|
|       103|    1|
|       120|    1|
|       107|    1|
|       102|    1|
|       105|    1|
|       110|    1|
|       104|    1|
+----------+-----+



In [97]:
from pyspark.sql.functions import sum

appointments_df.groupBy("patient_id") \
    .agg(
        sum("consultation_fee").alias("total_fee")
    ) \
    .show()

+----------+---------+
|patient_id|total_fee|
+----------+---------+
|       108|        0|
|       101|     2500|
|       103|     1500|
|       120|     1500|
|       107|     2000|
|       102|     2000|
|       105|     1000|
|       110|     2500|
|       104|     2500|
+----------+---------+



In [98]:
from pyspark.sql.functions import sum, desc

appointments_df.groupBy("patient_id") \
    .agg(
        sum("consultation_fee").alias("total_fee")
    ) \
    .orderBy(desc("total_fee")) \
    .show(1)

+----------+---------+
|patient_id|total_fee|
+----------+---------+
|       110|     2500|
+----------+---------+
only showing top 1 row


In [99]:
appointments_df.groupBy("patient_id") \
    .count() \
    .withColumnRenamed("count", "appointment_count") \
    .show()

+----------+-----------------+
|patient_id|appointment_count|
+----------+-----------------+
|       108|                1|
|       101|                2|
|       103|                1|
|       120|                1|
|       107|                1|
|       102|                1|
|       105|                1|
|       110|                1|
|       104|                1|
+----------+-----------------+



In [100]:
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    sum, rank, dense_rank, row_number,
    lead, lag, col
)

In [101]:
patient_fee_df = appointments_df.groupBy("patient_id") \
    .agg(sum("consultation_fee").alias("total_fee"))

In [102]:
window_spec = Window.orderBy(col("total_fee").desc())

patient_fee_df.withColumn(
    "rank",
    rank().over(window_spec)
).show()

+----------+---------+----+
|patient_id|total_fee|rank|
+----------+---------+----+
|       101|     2500|   1|
|       110|     2500|   1|
|       104|     2500|   1|
|       107|     2000|   4|
|       102|     2000|   4|
|       103|     1500|   6|
|       120|     1500|   6|
|       105|     1000|   8|
|       108|        0|   9|
+----------+---------+----+



In [103]:
patient_fee_df.withColumn(
    "dense_rank",
    dense_rank().over(window_spec)
).show()

+----------+---------+----------+
|patient_id|total_fee|dense_rank|
+----------+---------+----------+
|       101|     2500|         1|
|       110|     2500|         1|
|       104|     2500|         1|
|       107|     2000|         2|
|       102|     2000|         2|
|       103|     1500|         3|
|       120|     1500|         3|
|       105|     1000|         4|
|       108|        0|         5|
+----------+---------+----------+



In [104]:
patient_fee_df.withColumn(
    "row_number",
    row_number().over(window_spec)
).show()

+----------+---------+----------+
|patient_id|total_fee|row_number|
+----------+---------+----------+
|       101|     2500|         1|
|       110|     2500|         2|
|       104|     2500|         3|
|       107|     2000|         4|
|       102|     2000|         5|
|       103|     1500|         6|
|       120|     1500|         7|
|       105|     1000|         8|
|       108|        0|         9|
+----------+---------+----------+



In [105]:
patient_fee_df.orderBy(
    col("total_fee").desc()
).show(1)

+----------+---------+
|patient_id|total_fee|
+----------+---------+
|       110|     2500|
+----------+---------+
only showing top 1 row


In [106]:
patient_fee_df.orderBy(
    col("total_fee").desc()
).show(3)

+----------+---------+
|patient_id|total_fee|
+----------+---------+
|       101|     2500|
|       110|     2500|
|       104|     2500|
+----------+---------+
only showing top 3 rows


In [107]:
patient_city_fee = patient_fee_df.join(
    patients_df,
    on="patient_id"
)

window_city = Window.partitionBy("city").orderBy(
    col("total_fee").desc()
)

patient_city_fee.withColumn(
    "rank",
    rank().over(window_city)
).filter(
    col("rank") == 1
).show()

+----------+---------+------------+---------+---+------+-----------+----------------+-------------------+---------+--------------+--------------+----+
|patient_id|total_fee|patient_name|     city|age|gender|blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|rank|
+----------+---------+------------+---------+---+------+-----------+----------------+-------------------+---------+--------------+--------------+----+
|       110|     2500| Nisha Reddy|BANGALORE| 41|Female|         A+|          Active|           Complete|    Adult|             1|            No|   1|
|       104|     2500| Sneha Patel|  CHENNAI| 31|Female|         O+|          Active|           Complete|    Adult|             1|            No|   1|
|       105|     1000|  Farhan Ali|    DELHI| 55|  Male|        AB+|          Active|           Complete|   Senior|             1|            No|   1|
|       101|     2500|Rahul Sharma|HYDERABAD| 35|  Male|         O+|          Active|         

In [108]:
window_city = Window.partitionBy("city").orderBy(
    col("total_fee").asc()
)

patient_city_fee.withColumn(
    "rank",
    rank().over(window_city)
).filter(
    col("rank") == 1
).show()

+----------+---------+------------+---------+---+------+-----------+----------------+-------------------+---------+--------------+--------------+----+
|patient_id|total_fee|patient_name|     city|age|gender|blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|rank|
+----------+---------+------------+---------+---+------+-----------+----------------+-------------------+---------+--------------+--------------+----+
|       102|     2000| Priya Reddy|BANGALORE| 29|Female|         A+|          Active|           Complete|    Young|             1|            No|   1|
|       104|     2500| Sneha Patel|  CHENNAI| 31|Female|         O+|          Active|           Complete|    Adult|             1|            No|   1|
|       105|     1000|  Farhan Ali|    DELHI| 55|  Male|        AB+|          Active|           Complete|   Senior|             1|            No|   1|
|       101|     2500|Rahul Sharma|HYDERABAD| 35|  Male|         O+|          Active|         

In [109]:
window_spec = Window.orderBy("patient_id")

patient_fee_df.withColumn(
    "running_total",
    sum("total_fee").over(window_spec)
).show()

+----------+---------+-------------+
|patient_id|total_fee|running_total|
+----------+---------+-------------+
|       101|     2500|         2500|
|       102|     2000|         4500|
|       103|     1500|         6000|
|       104|     2500|         8500|
|       105|     1000|         9500|
|       107|     2000|        11500|
|       108|        0|        11500|
|       110|     2500|        14000|
|       120|     1500|        15500|
+----------+---------+-------------+



In [110]:
window_spec = Window.orderBy("patient_id")

patient_fee_df.withColumn(
    "running_total",
    sum("total_fee").over(window_spec)
).show()

+----------+---------+-------------+
|patient_id|total_fee|running_total|
+----------+---------+-------------+
|       101|     2500|         2500|
|       102|     2000|         4500|
|       103|     1500|         6000|
|       104|     2500|         8500|
|       105|     1000|         9500|
|       107|     2000|        11500|
|       108|        0|        11500|
|       110|     2500|        14000|
|       120|     1500|        15500|
+----------+---------+-------------+



In [111]:
window_spec = Window.orderBy("total_fee")

patient_fee_df.withColumn(
    "next_fee",
    lead("total_fee").over(window_spec)
).show()

+----------+---------+--------+
|patient_id|total_fee|next_fee|
+----------+---------+--------+
|       108|        0|    1000|
|       105|     1000|    1500|
|       103|     1500|    1500|
|       120|     1500|    2000|
|       107|     2000|    2000|
|       102|     2000|    2500|
|       101|     2500|    2500|
|       110|     2500|    2500|
|       104|     2500|    NULL|
+----------+---------+--------+



In [112]:
window_spec = Window.orderBy("total_fee")

patient_fee_df.withColumn(
    "previous_fee",
    lag("total_fee").over(window_spec)
).show()

+----------+---------+------------+
|patient_id|total_fee|previous_fee|
+----------+---------+------------+
|       108|        0|        NULL|
|       105|     1000|           0|
|       103|     1500|        1000|
|       120|     1500|        1500|
|       107|     2000|        1500|
|       102|     2000|        2000|
|       101|     2500|        2000|
|       110|     2500|        2500|
|       104|     2500|        2500|
+----------+---------+------------+



In [114]:
preferences_df.printSchema()

root
 |-- _corrupt_record: string (nullable = true)



In [115]:
%%writefile patient_preferences.json
[
  {
    "patient_id": 101,
    "preferred_hospital": "Apollo",
    "contact": {
      "phone": "9876500011",
      "email": "rahul@gmail.com"
    }
  },
  {
    "patient_id": 102,
    "preferred_hospital": "Yashoda",
    "contact": {
      "phone": null,
      "email": "priya@gmail.com"
    }
  },
  {
    "patient_id": 103,
    "preferred_hospital": "Care",
    "contact": {
      "phone": "9876500013",
      "email": null
    }
  },
  {
    "patient_id": 104,
    "preferred_hospital": null,
    "contact": {
      "phone": "9876500014",
      "email": "sneha@gmail.com"
    }
  }
]

Overwriting patient_preferences.json


In [116]:
preferences_df = spark.read.option(
    "multiline", "true"
).json("patient_preferences.json")

In [117]:
preferences_df.printSchema()
preferences_df.show(truncate=False)

root
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- patient_id: long (nullable = true)
 |-- preferred_hospital: string (nullable = true)

+-----------------------------+----------+------------------+
|contact                      |patient_id|preferred_hospital|
+-----------------------------+----------+------------------+
|{rahul@gmail.com, 9876500011}|101       |Apollo            |
|{priya@gmail.com, NULL}      |102       |Yashoda           |
|{NULL, 9876500013}           |103       |Care              |
|{sneha@gmail.com, 9876500014}|104       |NULL              |
+-----------------------------+----------+------------------+



In [119]:
preferences_df = spark.read.option(
    "multiline", "true"
).json("patient_preferences.json")

In [120]:
preferences_df.printSchema()
preferences_df.show(truncate=False)

root
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- patient_id: long (nullable = true)
 |-- preferred_hospital: string (nullable = true)

+-----------------------------+----------+------------------+
|contact                      |patient_id|preferred_hospital|
+-----------------------------+----------+------------------+
|{rahul@gmail.com, 9876500011}|101       |Apollo            |
|{priya@gmail.com, NULL}      |102       |Yashoda           |
|{NULL, 9876500013}           |103       |Care              |
|{sneha@gmail.com, 9876500014}|104       |NULL              |
+-----------------------------+----------+------------------+



In [122]:
preferences_df = spark.read.option(
    "multiline", "true"
).json("patient_preferences.json")

preferences_df.printSchema()
preferences_df.show(truncate=False)

root
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- patient_id: long (nullable = true)
 |-- preferred_hospital: string (nullable = true)

+-----------------------------+----------+------------------+
|contact                      |patient_id|preferred_hospital|
+-----------------------------+----------+------------------+
|{rahul@gmail.com, 9876500011}|101       |Apollo            |
|{priya@gmail.com, NULL}      |102       |Yashoda           |
|{NULL, 9876500013}           |103       |Care              |
|{sneha@gmail.com, 9876500014}|104       |NULL              |
+-----------------------------+----------+------------------+



In [124]:
preferences_df = spark.read.option(
    "multiline", "true"
).json("patient_preferences.json")

preferences_df.show(truncate=False)

+-----------------------------+----------+------------------+
|contact                      |patient_id|preferred_hospital|
+-----------------------------+----------+------------------+
|{rahul@gmail.com, 9876500011}|101       |Apollo            |
|{priya@gmail.com, NULL}      |102       |Yashoda           |
|{NULL, 9876500013}           |103       |Care              |
|{sneha@gmail.com, 9876500014}|104       |NULL              |
+-----------------------------+----------+------------------+



In [125]:
preferences_df.printSchema()

root
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- patient_id: long (nullable = true)
 |-- preferred_hospital: string (nullable = true)



In [126]:
preferences_df.select(
    "patient_id",
    "preferred_hospital",
    col("contact.phone").alias("phone")
).show()

+----------+------------------+----------+
|patient_id|preferred_hospital|     phone|
+----------+------------------+----------+
|       101|            Apollo|9876500011|
|       102|           Yashoda|      NULL|
|       103|              Care|9876500013|
|       104|              NULL|9876500014|
+----------+------------------+----------+



In [127]:
preferences_df.select(
    "patient_id",
    "preferred_hospital",
    col("contact.email").alias("email")
).show()

+----------+------------------+---------------+
|patient_id|preferred_hospital|          email|
+----------+------------------+---------------+
|       101|            Apollo|rahul@gmail.com|
|       102|           Yashoda|priya@gmail.com|
|       103|              Care|           NULL|
|       104|              NULL|sneha@gmail.com|
+----------+------------------+---------------+



In [128]:
preferences_df.filter(
    col("contact.phone").isNull()
).show(truncate=False)

+-----------------------+----------+------------------+
|contact                |patient_id|preferred_hospital|
+-----------------------+----------+------------------+
|{priya@gmail.com, NULL}|102       |Yashoda           |
+-----------------------+----------+------------------+



In [129]:
preferences_df.filter(
    col("contact.email").isNull()
).show(truncate=False)

+------------------+----------+------------------+
|contact           |patient_id|preferred_hospital|
+------------------+----------+------------------+
|{NULL, 9876500013}|103       |Care              |
+------------------+----------+------------------+



In [130]:
preferences_df.filter(
    col("preferred_hospital").isNull()
).show(truncate=False)

+-----------------------------+----------+------------------+
|contact                      |patient_id|preferred_hospital|
+-----------------------------+----------+------------------+
|{sneha@gmail.com, 9876500014}|104       |NULL              |
+-----------------------------+----------+------------------+



In [131]:
preferences_df.filter(
    col("preferred_hospital").isNull()
).show(truncate=False)

+-----------------------------+----------+------------------+
|contact                      |patient_id|preferred_hospital|
+-----------------------------+----------+------------------+
|{sneha@gmail.com, 9876500014}|104       |NULL              |
+-----------------------------+----------+------------------+



In [133]:
from pyspark.sql.functions import coalesce, lit

preferences_flat = preferences_df.select(
    "patient_id",
    "preferred_hospital",
    coalesce(col("contact.phone"), lit("Not Available")).alias("phone"),
    col("contact.email").alias("email")
)

preferences_flat.show()

+----------+------------------+-------------+---------------+
|patient_id|preferred_hospital|        phone|          email|
+----------+------------------+-------------+---------------+
|       101|            Apollo|   9876500011|rahul@gmail.com|
|       102|           Yashoda|Not Available|priya@gmail.com|
|       103|              Care|   9876500013|           NULL|
|       104|              NULL|   9876500014|sneha@gmail.com|
+----------+------------------+-------------+---------------+



In [134]:
from pyspark.sql.functions import coalesce, lit

preferences_flat = preferences_df.select(
    "patient_id",
    "preferred_hospital",
    col("contact.phone").alias("phone"),
    coalesce(col("contact.email"), lit("Not Available")).alias("email")
)

preferences_flat.show()

+----------+------------------+----------+---------------+
|patient_id|preferred_hospital|     phone|          email|
+----------+------------------+----------+---------------+
|       101|            Apollo|9876500011|rahul@gmail.com|
|       102|           Yashoda|      NULL|priya@gmail.com|
|       103|              Care|9876500013|  Not Available|
|       104|              NULL|9876500014|sneha@gmail.com|
+----------+------------------+----------+---------------+



In [135]:
preferences_flat = preferences_df.select(
    "patient_id",
    "preferred_hospital",
    col("contact.phone").alias("phone"),
    col("contact.email").alias("email")
)

patients_preferences = patients_df.join(
    preferences_flat,
    on="patient_id",
    how="left"
)

patients_preferences.show(truncate=False)

+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+------------------+----------+---------------+
|patient_id|patient_name|city     |age|gender|blood_group  |insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|preferred_hospital|phone     |email          |
+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+------------------+----------+---------------+
|101       |Rahul Sharma|HYDERABAD|35 |Male  |O+           |Active          |Complete           |Adult    |1             |No            |Apollo            |9876500011|rahul@gmail.com|
|102       |Priya Reddy |BANGALORE|29 |Female|A+           |Active          |Complete           |Young    |1             |No            |Yashoda           |NULL      |priya@gmail.com|
|103       |Amit Kumar  |MUMBAI   |42 |Male  |B+           |Inactive        |Com

In [136]:
patients_df.createOrReplaceTempView("patients")

In [137]:
appointments_df.createOrReplaceTempView("appointments")

In [138]:
spark.sql("""
SELECT *
FROM patients
""").show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|
+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+
|       101|Rahul Sharma|HYDERABAD| 35|  Male|           O+|          Active|           Complete|    Adult|             1|            No|
|       102| Priya Reddy|BANGALORE| 29|Female|           A+|          Active|           Complete|    Young|             1|            No|
|       103|  Amit Kumar|   MUMBAI| 42|  Male|           B+|        Inactive|           Complete|    Adult|             0|            No|
|       104| Sneha Patel|  CHENNAI| 31|Female|           O+|          Active|           Complete|    Adult|             1|            No|
|       105|  Farhan Ali|    DELHI

In [139]:
spark.sql("""
SELECT *
FROM patients
""").show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|
+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+
|       101|Rahul Sharma|HYDERABAD| 35|  Male|           O+|          Active|           Complete|    Adult|             1|            No|
|       102| Priya Reddy|BANGALORE| 29|Female|           A+|          Active|           Complete|    Young|             1|            No|
|       103|  Amit Kumar|   MUMBAI| 42|  Male|           B+|        Inactive|           Complete|    Adult|             0|            No|
|       104| Sneha Patel|  CHENNAI| 31|Female|           O+|          Active|           Complete|    Adult|             1|            No|
|       105|  Farhan Ali|    DELHI

In [140]:
spark.sql("""
SELECT *
FROM patients
WHERE city = 'Hyderabad'
""").show()

+----------+------------+----+---+------+-----------+----------------+-------------------+---------+--------------+--------------+
|patient_id|patient_name|city|age|gender|blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|
+----------+------------+----+---+------+-----------+----------------+-------------------+---------+--------------+--------------+
+----------+------------+----+---+------+-----------+----------------+-------------------+---------+--------------+--------------+



In [141]:
spark.sql("""
SELECT city,
       COUNT(*) AS patient_count
FROM patients
GROUP BY city
""").show()

+---------+-------------+
|     city|patient_count|
+---------+-------------+
|    KOCHI|            1|
|BANGALORE|            2|
|    DELHI|            1|
|HYDERABAD|            2|
|  UNKNOWN|            1|
|  CHENNAI|            1|
|     PUNE|            1|
|   MUMBAI|            1|
+---------+-------------+



In [142]:
spark.sql("""
SELECT department,
       COUNT(*) AS appointment_count
FROM appointments
GROUP BY department
""").show()

+-----------+-----------------+
| department|appointment_count|
+-----------+-----------------+
|  Neurology|                2|
|Dermatology|                3|
| Cardiology|                3|
|Orthopedics|                2|
+-----------+-----------------+



In [143]:
spark.sql("""
SELECT department,
       COUNT(*) AS appointment_count
FROM appointments
GROUP BY department
""").show()

+-----------+-----------------+
| department|appointment_count|
+-----------+-----------------+
|  Neurology|                2|
|Dermatology|                3|
| Cardiology|                3|
|Orthopedics|                2|
+-----------+-----------------+



In [144]:
spark.sql("""
SELECT department,
       AVG(consultation_fee) AS average_fee
FROM appointments
GROUP BY department
""").show()

+-----------+-----------------+
| department|      average_fee|
+-----------+-----------------+
|  Neurology|           2000.0|
|Dermatology|666.6666666666666|
| Cardiology|           1500.0|
|Orthopedics|           2500.0|
+-----------+-----------------+



In [145]:
spark.sql("""
SELECT MAX(consultation_fee) AS highest_fee
FROM appointments
""").show()

+-----------+
|highest_fee|
+-----------+
|       2500|
+-----------+



In [146]:
spark.sql("""
SELECT patient_id,
       COUNT(*) AS appointment_count
FROM appointments
GROUP BY patient_id
ORDER BY patient_id
""").show()

+----------+-----------------+
|patient_id|appointment_count|
+----------+-----------------+
|       101|                2|
|       102|                1|
|       103|                1|
|       104|                1|
|       105|                1|
|       107|                1|
|       108|                1|
|       110|                1|
|       120|                1|
+----------+-----------------+



In [147]:
spark.sql("""
SELECT patient_id,
       SUM(consultation_fee) AS total_fee
FROM appointments
GROUP BY patient_id
ORDER BY total_fee DESC
LIMIT 5
""").show()

+----------+---------+
|patient_id|total_fee|
+----------+---------+
|       101|     2500|
|       110|     2500|
|       104|     2500|
|       107|     2000|
|       102|     2000|
+----------+---------+



In [148]:
patients_df = spark.read.csv(
    "patients.csv",
    header=True,
    inferSchema=True
)

appointments_df = spark.read.csv(
    "appointments.csv",
    header=True,
    inferSchema=True
)

In [149]:
preferences_df = spark.read.option(
    "multiline",
    "true"
).json("patient_preferences.json")

In [150]:
from pyspark.sql.functions import col

patients_df = patients_df.fillna({
    "city": "Unknown",
    "blood_group": "Not Available"
})

appointments_df = appointments_df.fillna({
    "consultation_fee": 0
})

preferences_flat = preferences_df.select(
    "patient_id",
    "preferred_hospital",
    col("contact.phone").alias("phone"),
    col("contact.email").alias("email")
).fillna({
    "phone": "Not Available",
    "email": "Not Available",
    "preferred_hospital": "Unknown"
})

In [151]:
hospital_df = patients_df.join(
    appointments_df,
    on="patient_id",
    how="left"
).join(
    preferences_flat,
    on="patient_id",
    how="left"
)

hospital_df.show()

+----------+------------+---------+---+------+-------------+----------------+--------------+-----------+-----------+----------------+----------------+---------+------------------+-------------+---------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|appointment_id|doctor_name| department|appointment_date|consultation_fee|   status|preferred_hospital|        phone|          email|
+----------+------------+---------+---+------+-------------+----------------+--------------+-----------+-----------+----------------+----------------+---------+------------------+-------------+---------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|          5003|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|            Apollo|   9876500011|rahul@gmail.com|
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|          5001| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|

In [152]:
from pyspark.sql.functions import when

hospital_df = hospital_df.withColumn(
    "age_group",
    when(col("age") < 30, "Young")
    .when(col("age") < 50, "Adult")
    .otherwise("Senior")
)

hospital_df.show()

+----------+------------+---------+---+------+-------------+----------------+--------------+-----------+-----------+----------------+----------------+---------+------------------+-------------+---------------+---------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|appointment_id|doctor_name| department|appointment_date|consultation_fee|   status|preferred_hospital|        phone|          email|age_group|
+----------+------------+---------+---+------+-------------+----------------+--------------+-----------+-----------+----------------+----------------+---------+------------------+-------------+---------------+---------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|          5003|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|            Apollo|   9876500011|rahul@gmail.com|    Adult|
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|          5001| Dr. Ramesh| Cardiology|    

In [153]:
from pyspark.sql.functions import sum

hospital_df.groupBy(
    "department"
).agg(
    sum("consultation_fee").alias("department_revenue")
).show()

+-----------+------------------+
| department|department_revenue|
+-----------+------------------+
|       NULL|              NULL|
|  Neurology|              4000|
|Dermatology|              2000|
| Cardiology|              3000|
|Orthopedics|              5000|
+-----------+------------------+



In [154]:
hospital_df.groupBy(
    "patient_id",
    "patient_name"
).agg(
    sum("consultation_fee").alias("total_spending")
).show()

+----------+------------+--------------+
|patient_id|patient_name|total_spending|
+----------+------------+--------------+
|       107| Arjun Verma|          2000|
|       108|  Meera Nair|             0|
|       109|   Kiran Rao|          NULL|
|       110| Nisha Reddy|          2500|
|       105|  Farhan Ali|          1000|
|       101|Rahul Sharma|          2500|
|       104| Sneha Patel|          2500|
|       103|  Amit Kumar|          1500|
|       106|  Neha Singh|          NULL|
|       102| Priya Reddy|          2000|
+----------+------------+--------------+



In [155]:
hospital_df.groupBy(
    "department"
).agg(
    sum("consultation_fee").alias("total_revenue")
).show()

+-----------+-------------+
| department|total_revenue|
+-----------+-------------+
|       NULL|         NULL|
|  Neurology|         4000|
|Dermatology|         2000|
| Cardiology|         3000|
|Orthopedics|         5000|
+-----------+-------------+



In [156]:
hospital_df.groupBy(
    "department"
).agg(
    sum("consultation_fee").alias("total_revenue")
).show()

+-----------+-------------+
| department|total_revenue|
+-----------+-------------+
|       NULL|         NULL|
|  Neurology|         4000|
|Dermatology|         2000|
| Cardiology|         3000|
|Orthopedics|         5000|
+-----------+-------------+



In [157]:
hospital_df.write.mode(
    "overwrite"
).parquet("hospital_analytics_output")

In [158]:
print("========== HOSPITAL ANALYTICS REPORT ==========")

print("Total Patients:",
      patients_df.count())

print("Total Appointments:",
      appointments_df.count())

print("Total Revenue:")
hospital_df.groupBy().sum(
    "consultation_fee"
).show()

print("Department Revenue:")
hospital_df.groupBy(
    "department"
).sum(
    "consultation_fee"
).show()

print("Patient-wise Spending:")
hospital_df.groupBy(
    "patient_name"
).sum(
    "consultation_fee"
).show()

print("Age Group Distribution:")
hospital_df.groupBy(
    "age_group"
).count().show()

print("==============================================")

========== HOSPITAL ANALYTICS REPORT ==========
Total Patients: 10
Total Appointments: 10
Total Revenue:
+---------------------+
|sum(consultation_fee)|
+---------------------+
|                14000|
+---------------------+

Department Revenue:
+-----------+---------------------+
| department|sum(consultation_fee)|
+-----------+---------------------+
|       NULL|                 NULL|
|  Neurology|                 4000|
|Dermatology|                 2000|
| Cardiology|                 3000|
|Orthopedics|                 5000|
+-----------+---------------------+

Patient-wise Spending:
+------------+---------------------+
|patient_name|sum(consultation_fee)|
+------------+---------------------+
|  Amit Kumar|                 1500|
|   Kiran Rao|                 NULL|
|Rahul Sharma|                 2500|
|  Meera Nair|                    0|
| Sneha Patel|                 2500|
|  Farhan Ali|                 1000|
| Arjun Verma|                 2000|
|  Neha Singh|                 NULL|